# Tutorial 3: Hardware Acceleration Layer (HAL)

This notebook introduces the **SMGP Hardware Abstraction Layer (HAL)**, which
provides transparent hardware acceleration for SMGP operations.

The HAL architecture includes:
- A custom ISA (Instruction Set Architecture) for graph operations.
- A hardware executor that translates Python API calls to ISA instructions.
- Memory-mapped I/O for efficient data transfer between host and accelerator.

**References:**
- Hennessy, J.L. & Patterson, D.A. (2019). "Computer Architecture: A Quantitative
  Approach." 6th Ed. Morgan Kaufmann.
- SMGP Hardware Architecture Documentation (`hardware/docs_hw/architecture.md`).

In [1]:
import sys
import os

# Add the hardware HAL to the Python path
hw_hal_path = os.path.join(os.getcwd(), "hardware", "sw")
if hw_hal_path not in sys.path:
    sys.path.insert(0, hw_hal_path)

import numpy as np
np.set_printoptions(precision=4, suppress=True)

print("Hardware HAL path added to sys.path.")

Hardware HAL path added to sys.path.


## 1. Hardware Architecture Overview

The SMGP accelerator implements the following compute engines in hardware:

| Engine | Function | ISA Opcode |
|--------|----------|------------|
| Topology Engine | Graph construction, node/edge management | `0x1` |
| Spectral Engine | Laplacian computation, eigen-decomposition, Chebyshev convolution | `0x2` |
| HD Engine | Bind, unbind, bundle, similarity of hyperdimensional vectors | `0x3` |
| Topology Analyzer | Filtration, persistence, stability checking | `0x4` |
| Graph Rewrite Engine | DPO graph rewriting rules | `0x5` |
| Memory Controller | Associative cache, HBM management | `0x6` |

The accelerator communicates with the host via memory-mapped I/O through
a Network-on-Chip (NoC) interconnect.

In [2]:
# ISA Instruction Encoding
# Each instruction is 32 bits: [opcode:4][sub_opcode:4][flags:8][operand:16]

def encode_instruction(opcode: int, sub_opcode: int, flags: int, operand: int) -> int:
    """Encode a 32-bit SMGP ISA instruction.
    
    Bit layout:
      [31:28] opcode      (4 bits) - operation category
      [27:24] sub_opcode  (4 bits) - specific operation
      [23:16] flags       (8 bits) - control flags
      [15:0]  operand     (16 bits) - data/address operand
    """
    return (
        ((opcode & 0xF) << 28) |
        ((sub_opcode & 0xF) << 24) |
        ((flags & 0xFF) << 16) |
        (operand & 0xFFFF)
    )

# Example: ADD_NODE instruction
# opcode=0x1 (GRAPH_CTOR), sub_opcode=0x0 (ADD_NODE), flags=0x01 (START), operand=0
add_node_instr = encode_instruction(0x1, 0x0, 0x01, 0)
print(f"ADD_NODE instruction (hex):  0x{add_node_instr:08X}")
print(f"ADD_NODE instruction (bin):  0b{add_node_instr:032b}")

# Example: COMPUTE_LAPLACIAN instruction
# opcode=0x2 (SPECTRAL), sub_opcode=0x0 (COMPUTE_LAP), flags=0x01 (START), operand=16
compute_lap_instr = encode_instruction(0x2, 0x0, 0x01, 16)
print(f"COMPUTE_LAP instruction (hex): 0x{compute_lap_instr:08X}")

# Example: HD_BIND instruction
hd_bind_instr = encode_instruction(0x3, 0x1, 0x01, 0)
print(f"HD_BIND instruction (hex):     0x{hd_bind_instr:08X}")

ADD_NODE instruction (hex):  0x10010000
ADD_NODE instruction (bin):  0b00010000000000010000000000000000
COMPUTE_LAP instruction (hex): 0x20010010
HD_BIND instruction (hex):     0x31010000


## 2. Creating a Hardware Session (Mock/No-Op Demo)

The `HWSession` manages the connection to the hardware accelerator.
In simulation or demo mode, it provides no-op execution that mimics
the hardware interface without requiring actual FPGA/ASIC hardware.

The `HWExecutor` translates high-level SMGP operations (add_node,
compute_laplacian, hd_bind, etc.) into ISA instructions automatically.

In [3]:
# Create a mock hardware session for demonstration
# (In production, this connects to the FPGA/ASIC via PCIe driver)

class MockHWSession:
    """Mock hardware session for demonstration purposes.
    
    Simulates the hardware interface without requiring actual hardware.
    Tracks executed instructions for inspection.
    """
    
    def __init__(self, backend: str = "mock"):
        self.backend = backend
        self.executed_instructions = []
        self.registers = {}
    
    def execute(self, instruction: int) -> bool:
        """Execute a single ISA instruction (mock implementation)."""
        self.executed_instructions.append(instruction)
        return True
    
    def read_reg(self, addr: int) -> int | None:
        """Read a hardware register (mock implementation)."""
        return self.registers.get(addr, 0)
    
    def write_reg(self, addr: int, value: int) -> None:
        """Write a hardware register (mock implementation)."""
        self.registers[addr] = value


session = MockHWSession(backend="mock")
print(f"Created mock hardware session: backend={session.backend}")
print(f"Session ready for instruction execution.")

Created mock hardware session: backend=mock
Session ready for instruction execution.


In [4]:
# Create an HWExecutor that translates API calls to ISA instructions
# We implement a simplified version for the demo

class DemoHWExecutor:
    """Simplified hardware executor for demonstration.
    
    Mirrors the interface of smgp.core.graph.SpectralMemoryGraph but
    offloads computation to the SMGP accelerator via ISA instructions.
    """
    
    # ISA opcodes
    OPC_GRAPH = 0x1
    OPC_SPECTRAL = 0x2
    OPC_HD = 0x3
    OPC_TOPOLOGY = 0x4
    
    def __init__(self, session: MockHWSession):
        self.session = session
        self.node_count = 0
        self.nodes = {}  # node_id -> hd_vector
    
    def add_node(self, node_id: str, vector: np.ndarray | None = None) -> str:
        """Add a node via hardware."""
        instr = encode_instruction(
            self.OPC_GRAPH, 0x0, 0x01,  # ADD_NODE
            self.node_count & 0xFFFF
        )
        self.session.execute(instr)
        
        if vector is not None:
            self.nodes[node_id] = vector
        self.node_count += 1
        return node_id
    
    def add_edge(self, source: str, target: str, relation: str = "") -> str:
        """Add an edge via hardware."""
        instr = encode_instruction(
            self.OPC_GRAPH, 0x1, 0x01,  # ADD_EDGE
            hash((source, target)) & 0xFFFF
        )
        self.session.execute(instr)
        return f"edge_{source}_{target}"
    
    def compute_laplacian(self) -> bool:
        """Offload Laplacian computation to hardware."""
        instr = encode_instruction(
            self.OPC_SPECTRAL, 0x0, 0x01,  # COMPUTE_LAP
            self.node_count & 0xFFFF
        )
        return self.session.execute(instr)
    
    def hd_bind(self, a: np.ndarray, b: np.ndarray) -> np.ndarray:
        """Offload HD bind to hardware (element-wise XOR for bipolar)."""
        instr = encode_instruction(self.OPC_HD, 0x1, 0x01, 0)
        self.session.execute(instr)
        # Hardware computes element-wise product (bipolar XOR)
        return np.where(a == b, 1, -1).astype(np.int8)
    
    def hd_similarity(self, a: np.ndarray, b: np.ndarray) -> float:
        """Offload HD similarity to hardware."""
        instr = encode_instruction(self.OPC_HD, 0x4, 0x01, 0)
        self.session.execute(instr)
        return float(np.dot(a.astype(np.float64), b.astype(np.float64))) / len(a)


executor = DemoHWExecutor(session)
print(f"Created HWExecutor with {executor.node_count} initial nodes.")

Created HWExecutor with 0 initial nodes.


In [5]:
# Execute operations through the hardware executor
from smgp.core.hyperdim import HyperdimensionalMemory

hd = HyperdimensionalMemory(dim=1000, seed=42)

# Add nodes through hardware
for i in range(5):
    vec = hd.generate(1)[0]
    executor.add_node(f"hw_node_{i}", vector=vec)

print(f"Nodes added via hardware: {executor.node_count}")
print(f"Instructions executed: {len(session.executed_instructions)}")

# Add edges
for i in range(4):
    executor.add_edge(f"hw_node_{i}", f"hw_node_{i+1}", "connected")

# Compute Laplacian on hardware
success = executor.compute_laplacian()
print(f"Laplacian computed on hardware: {success}")
print(f"Total instructions executed: {len(session.executed_instructions)}")

# Show executed instructions
print("\nExecuted ISA instructions:")
for i, instr in enumerate(session.executed_instructions):
    opcode = (instr >> 28) & 0xF
    sub_op = (instr >> 24) & 0xF
    flags = (instr >> 16) & 0xFF
    operand = instr & 0xFFFF
    print(f"  [{i:2d}] opcode=0x{opcode:X} sub=0x{sub_op:X} "
          f"flags=0x{flags:02X} operand=0x{operand:04X}")

Nodes added via hardware: 5
Instructions executed: 5
Laplacian computed on hardware: True
Total instructions executed: 10

Executed ISA instructions:
  [ 0] opcode=0x1 sub=0x0 flags=0x01 operand=0x0000
  [ 1] opcode=0x1 sub=0x0 flags=0x01 operand=0x0001
  [ 2] opcode=0x1 sub=0x0 flags=0x01 operand=0x0002
  [ 3] opcode=0x1 sub=0x0 flags=0x01 operand=0x0003
  [ 4] opcode=0x1 sub=0x0 flags=0x01 operand=0x0004
  [ 5] opcode=0x1 sub=0x1 flags=0x01 operand=0x7FC8
  [ 6] opcode=0x1 sub=0x1 flags=0x01 operand=0x024A
  [ 7] opcode=0x1 sub=0x1 flags=0x01 operand=0xC15C
  [ 8] opcode=0x1 sub=0x1 flags=0x01 operand=0x4CF5
  [ 9] opcode=0x2 sub=0x0 flags=0x01 operand=0x0005


## 3. Performance Targets

The SMGP hardware accelerator targets the following performance metrics
compared to software-only execution:

| Operation | Software (CPU) | Hardware (FPGA) | Speedup |
|-----------|---------------|-----------------|---------|
| HD Bind (D=10000) | ~10 μs | ~0.1 μs | ~100x |
| HD Similarity (D=10000) | ~15 μs | ~0.1 μs | ~150x |
| HD Bundle (100 vectors) | ~50 μs | ~1 μs | ~50x |
| Laplacian (N=1000) | ~500 μs | ~10 μs | ~50x |
| Eigen-decomp (N=1000, K=64) | ~5 ms | ~50 μs | ~100x |
| Chebyshev Conv (K=5, N=1000) | ~200 μs | ~5 μs | ~40x |

These targets assume a Xilinx Zynq UltraScale+ FPGA running at 250 MHz
with HBM2 memory providing 256 GB/s bandwidth.

In [6]:
# Demonstrate HD operations through hardware executor
v1 = hd.generate(1)[0]
v2 = hd.generate(1)[0]

# Hardware HD bind
bound = executor.hd_bind(v1, v2)

# Hardware HD similarity
sim = executor.hd_similarity(v1, v2)

# Compare with software implementation
sw_bound = hd.bind(v1, v2)
sw_sim = hd.similarity(v1, v2)

print("Hardware vs Software comparison:")
print(f"  HD Bind match:  {np.array_equal(bound, sw_bound)}")
print(f"  HW similarity:  {sim:.6f}")
print(f"  SW similarity:  {sw_sim:.6f}")
print(f"\nNote: Hardware similarity uses dot-product / D, while software uses cosine.")
print(f"For bipolar vectors, these are closely related.")

Hardware vs Software comparison:
  HD Bind match:  True
  HW similarity:  0.000000
  SW similarity:  0.000000

Note: Hardware similarity uses dot-product / D, while software uses cosine.
For bipolar vectors, these are closely related.


## Summary

This notebook covered:

1. **ISA encoding**: The 32-bit instruction format with opcode, sub-opcode,
   flags, and operand fields.
2. **HWExecutor interface**: Translating high-level API calls to ISA instructions.
3. **Mock execution**: Demonstrating the hardware flow without physical hardware.
4. **Performance targets**: Expected speedups for HD operations, spectral
   analysis, and graph convolution.

The HAL allows transparent switching between software and hardware backends,
enabling the same SMGP code to run efficiently on CPUs, GPUs, and custom
FPGA/ASIC accelerators.